# Run M: Mask2Former (Swin-Tiny, ADE20K-semantic pretrained), binary defect segmentation

Mask2Former was published at CVPR 2022: *Masked-attention Mask Transformer for Universal Image Segmentation*. Unlike SegFormer/SegNeXt (simple per-pixel classifiers), Mask2Former is a **mask-classification** architecture -- it predicts a fixed set of mask queries, each with a class label, matched to ground truth via Hungarian matching (DETR-style, same matching idea as D-FINE/RT-DETRv2, just for segmentation instead of detection). It's genuinely instance-capable, but this run treats it the same as the other segmentation baselines here -- binary semantic segmentation, mAP left blank -- for consistency across the table. Flag it if you want the full instance-mode version instead (bigger build: needs per-instance ground truth derived from the bounding box labels, real COCO-style AP).

Uses HuggingFace transformers directly (`facebook/mask2former-swin-tiny-ade-semantic`, the smallest available Mask2Former checkpoint), same binary masks from `SmallDefectPreprocessing`, same fixed size-stratified split, early-stops on validation Dice, evaluates overall/small/medium/large test subsets.

**Parameters chosen, and why (see config cell for exact values):**
- Batch size dropped to 4 (from 8 for SegFormer/SegNeXt) -- Mask2Former's Swin backbone + pixel decoder + transformer decoder + 100-query Hungarian matching is meaningfully heavier than a simple per-pixel classifier.
- Differential learning rate (backbone vs. head) instead of SegFormer's flat single LR -- Mask2Former's own training recipe uses this, and the exact parameter-name split (`pixel_level_module.encoder` for the Swin backbone) was verified against this checkpoint's real parameter names before use, not guessed.
- **AMP (mixed precision) is off, full precision only** -- Mask2Former is DETR-style (Hungarian-matching loss), the same architecture family that produced a documented NaN-loss failure under AMP on this exact dataset during the D-FINE run. Not worth risking the same failure mode here to save time on a free Kaggle GPU.
- Training-time API (`mask_labels`/`class_labels` as variable-length lists, not simple tensors) and the inference/post-processing path were both verified locally against the real HF source and a live forward+backward+inference test before writing this notebook.

Kaggle setup: attach **SmallDefectPreprocessing** as an input, enable Internet, and use a GPU.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import os
import random
import shutil
import subprocess
import sys
import time

RUN_NAME = 'RunM_mask2former_swin_tiny_imgsz640'
MODEL_NAME = 'facebook/mask2former-swin-tiny-ade-semantic'
MODEL_LABEL = 'Mask2Former'
WANDB_PROJECT = 'smallDefectDetection'
WANDB_RUN_NAME = f'{MODEL_LABEL}_segmentation'
IMG_SIZE = 640
BATCH_SIZE = 4
MAX_EPOCHS = 50
PATIENCE = 15
BASE_LR = 1e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 0.05
NUM_WORKERS = 2
SEED = 42

DATASET_NAMES = ['DAGM', 'GC10-DET', 'KolektorSDD2', 'MPDD', 'MTD', 'Severstal', 'VisA']
SIZE_BUCKETS = ['small', 'medium', 'large']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
RUN_DIR = WORKING_ROOT / 'mask2former_runs' / RUN_NAME
FINAL_OUTPUT_DIR = WORKING_ROOT / 'final_outputs' / RUN_NAME

print({'run': RUN_NAME, 'model': MODEL_NAME, 'imgsz': IMG_SIZE, 'batch': BATCH_SIZE, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'base_lr': BASE_LR, 'backbone_lr': BACKBONE_LR})

In [ ]:
# Kaggle Internet must be enabled for this cell.
# transformers pinned <4.52 -- 4.52+ switched to a torch.distributed.tensor import path
# that only exists in PyTorch 2.5+, no fallback for older torch (confirmed via a real
# failure on a RunPod pod's PyTorch 2.4.0 template: ImportError on DTensor). Kaggle's
# torch is usually newer so this may not be strictly required here, but pinning both
# scripts the same way avoids relying on that.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.51.0,<4.52.0', 'safetensors', 'wandb'], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import wandb
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable a Kaggle GPU before training.')

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True
print('Using device:', torch.cuda.get_device_name(0))


def wandb_login_anywhere():
    # Works on RunPod (env var) and Kaggle (Secrets add-on) without ever hardcoding the key.
    api_key = os.environ.get('WANDB_API_KEY')
    if not api_key:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception:
            api_key = None
    if api_key:
        wandb.login(key=api_key)
    else:
        wandb.login()


wandb_login_anywhere()
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        'model': MODEL_NAME,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'base_lr': BASE_LR,
        'backbone_lr': BACKBONE_LR,
        'weight_decay': WEIGHT_DECAY,
        'seed': SEED,
        'amp': False,
    },
)

In [ ]:
# Find the normal SmallDefectPreprocessing Kaggle input automatically.
expected_datasets = set(DATASET_NAMES)
source_candidates = []

for root, dirs, _ in os.walk(KAGGLE_INPUT_ROOT):
    matches = expected_datasets.intersection(dirs)
    if len(matches) >= 5:
        source_candidates.append((len(matches), Path(root)))

if not source_candidates:
    raise FileNotFoundError(
        'Could not find the processed dataset. Attach SmallDefectPreprocessing as a Kaggle input.'
    )

source_candidates.sort(key=lambda item: (-item[0], len(str(item[1]))))
SOURCE_ROOT = source_candidates[0][1]
print('Using processed source:', SOURCE_ROOT)
print('Datasets:', sorted(path.name for path in SOURCE_ROOT.iterdir() if path.is_dir()))

In [ ]:
def index_files(directory, suffixes):
    return {
        path.stem: path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in suffixes
    }

def candidate_stems(image_stem, target):
    base = image_stem.removesuffix('_defect')
    if target == 'mask':
        return [image_stem, image_stem.replace('_defect', '_mask'), base, base + '_mask', base + '_gt']
    return [image_stem, image_stem.replace('_defect', '_bbs'), base, base + '_bbs']

samples = []
missing = []

for dataset_name in DATASET_NAMES:
    for size_bucket in SIZE_BUCKETS:
        bucket_root = SOURCE_ROOT / dataset_name / size_bucket
        image_dir = bucket_root / 'images'
        mask_dir = bucket_root / 'masks'
        label_dir = bucket_root / 'labels_yolo'

        if not image_dir.exists() or not mask_dir.exists() or not label_dir.exists():
            missing.append((dataset_name, size_bucket, 'missing directory'))
            continue

        mask_index = index_files(mask_dir, IMAGE_EXTS)
        label_index = index_files(label_dir, {'.txt'})
        matched = 0

        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue

            mask_path = next((mask_index[stem] for stem in candidate_stems(image_path.stem, 'mask') if stem in mask_index), None)
            label_path = next((label_index[stem] for stem in candidate_stems(image_path.stem, 'box') if stem in label_index), None)

            if mask_path is None or label_path is None:
                missing.append((dataset_name, size_bucket, image_path.name))
                continue

            samples.append({
                'image_path': image_path,
                'mask_path': mask_path,
                'dataset': dataset_name,
                'size': size_bucket,
                'stratum': dataset_name + '_' + size_bucket,
            })
            matched += 1

        print(f'{dataset_name}/{size_bucket}: {matched} matched')

print('Total matched:', len(samples))
print('Missing:', len(missing))
if len(samples) != 12670:
    raise RuntimeError(f'Expected 12,670 image-mask-label triplets, found {len(samples)}. First missing: {missing[:10]}')

In [ ]:
# Same fixed 70/15/15 stratified split as the other runs.
by_stratum = defaultdict(list)
for sample in samples:
    by_stratum[sample['stratum']].append(sample)

rng = random.Random(SEED)
train_samples, val_samples, test_samples = [], [], []
for _, group in sorted(by_stratum.items()):
    group = list(group)
    rng.shuffle(group)
    train_end = int(len(group) * 0.70)
    val_end = train_end + int(len(group) * 0.15)
    train_samples.extend(group[:train_end])
    val_samples.extend(group[train_end:val_end])
    test_samples.extend(group[val_end:])

rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)

test_sets = {
    'overall': test_samples,
    'small': [sample for sample in test_samples if sample['size'] == 'small'],
    'medium': [sample for sample in test_samples if sample['size'] == 'medium'],
    'large': [sample for sample in test_samples if sample['size'] == 'large'],
}

def print_counts(name, split):
    counts = Counter(sample['size'] for sample in split)
    print(f'{name}: total={len(split)}, small={counts["small"]}, medium={counts["medium"]}, large={counts["large"]}')

print_counts('Train', train_samples)
print_counts('Validation', val_samples)
for name, split in test_sets.items():
    print_counts('Test ' + name, split)

assert len(train_samples) == 8858
assert len(val_samples) == 1892
assert len(test_samples) == 1920

In [ ]:
def load_binary_mask(mask_path, target_size):
    mask = Image.open(mask_path).convert('L')
    if mask.size != target_size:
        mask = mask.resize(target_size, Image.Resampling.NEAREST)
    return (np.asarray(mask) > 0).astype(np.uint8)


# do_resize=False: images/masks are resized ourselves below, so the exact same array
# that produces mask_labels/class_labels (via the processor) is also kept as the raw
# ground-truth tensor for pixel-level metrics -- no risk of the processor's internal
# resize logic silently drifting from what's used for evaluation.
# do_reduce_labels=False: label 0 is a real 'background' class here, not 'ignore' --
# ADE20K's own convention (which this checkpoint was pretrained under) would otherwise
# treat class 0 as ignore and shift everything down by one, which is wrong for us.
processor = Mask2FormerImageProcessor.from_pretrained(
    MODEL_NAME,
    do_resize=False,
    do_reduce_labels=False,
)


class DefectMaskDataset(Dataset):
    def __init__(self, split_samples):
        self.samples = split_samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(sample['image_path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        mask = load_binary_mask(sample['mask_path'], (IMG_SIZE, IMG_SIZE))
        encoded = processor(images=image, segmentation_maps=mask, return_tensors='pt')
        return {
            'pixel_values': encoded['pixel_values'][0],
            'pixel_mask': encoded['pixel_mask'][0],
            'mask_labels': encoded['mask_labels'][0],
            'class_labels': encoded['class_labels'][0],
            'gt_semantic_mask': torch.from_numpy(mask).long(),
        }


def collate_fn(batch):
    # mask_labels/class_labels are variable-length per image (however many classes are
    # actually present) -- they can't be torch.stack'd like a normal batch, Mask2Former's
    # forward() expects them as plain lists. Verified this exact shape against the real
    # model API locally before using it here.
    return {
        'pixel_values': torch.stack([item['pixel_values'] for item in batch]),
        'pixel_mask': torch.stack([item['pixel_mask'] for item in batch]),
        'mask_labels': [item['mask_labels'] for item in batch],
        'class_labels': [item['class_labels'] for item in batch],
        'gt_semantic_mask': torch.stack([item['gt_semantic_mask'] for item in batch]),
    }


def make_loader(split_samples, shuffle=False):
    return DataLoader(
        DefectMaskDataset(split_samples),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        collate_fn=collate_fn,
    )


train_loader = make_loader(train_samples, shuffle=True)
val_loader = make_loader(val_samples)
print('Train batches:', len(train_loader), 'Validation batches:', len(val_loader))

In [ ]:
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
).to(DEVICE)

# Differential LR: lower for the pretrained Swin backbone, higher for the randomly
# reinitialized classification/mask heads -- Mask2Former's own training recipe uses this
# split, and 'pixel_level_module.encoder' was confirmed against this checkpoint's real
# parameter names (431 backbone / 328 head params) before writing this, not guessed.
backbone_params = [p for n, p in model.named_parameters() if 'pixel_level_module.encoder' in n]
head_params = [p for n, p in model.named_parameters() if 'pixel_level_module.encoder' not in n]
optimizer = AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': head_params, 'lr': BASE_LR},
], weight_decay=WEIGHT_DECAY)

RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Model loaded. Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print(f'Backbone params: {sum(p.numel() for p in backbone_params):,} (lr={BACKBONE_LR}), head params: {sum(p.numel() for p in head_params):,} (lr={BASE_LR})')

In [ ]:
@torch.inference_mode()
def evaluate(loader, measure_inference=False):
    model.eval()
    true_positive = false_positive = false_negative = 0
    inference_seconds = 0.0
    image_count = 0

    for batch in loader:
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        pixel_mask = batch['pixel_mask'].to(DEVICE, non_blocking=True)
        gt_masks = batch['gt_semantic_mask']

        if measure_inference:
            torch.cuda.synchronize()
            start = time.perf_counter()
        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
        if measure_inference:
            torch.cuda.synchronize()
            inference_seconds += time.perf_counter() - start

        target_sizes = [(IMG_SIZE, IMG_SIZE)] * pixel_values.shape[0]
        predictions = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)
        predictions = torch.stack(predictions).cpu()

        true_positive += int(((predictions == 1) & (gt_masks == 1)).sum().item())
        false_positive += int(((predictions == 1) & (gt_masks == 0)).sum().item())
        false_negative += int(((predictions == 0) & (gt_masks == 1)).sum().item())
        image_count += gt_masks.shape[0]

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    iou = true_positive / max(true_positive + false_positive + false_negative, 1)
    dice = 2 * true_positive / max(2 * true_positive + false_positive + false_negative, 1)

    return {
        'precision': precision,
        'recall': recall,
        'iou': iou,
        'dice': dice,
        # Pixel-level FPs per image, not per-instance -- kept identical in spirit to
        # SegFormer/SegNeXt's fp_per_image even though this model is instance-capable,
        # since it's being evaluated in semantic mode here (see markdown cell).
        'fp_per_image': false_positive / max(image_count, 1),
        'inference_time_ms_per_image': 1000 * inference_seconds / max(image_count, 1),
        'images': image_count,
    }


history = []
best_dice = -1.0
best_epoch = -1
epochs_without_improvement = 0
PROGRESS_EVERY = 50  # batches -- this loop used to print nothing until a full epoch
                      # finished, which on a slow GPU looks indistinguishable from a hang.

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    epoch_start = time.perf_counter()

    for batch_idx, batch in enumerate(train_loader, start=1):
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        pixel_mask = batch['pixel_mask'].to(DEVICE, non_blocking=True)
        mask_labels = [m.to(DEVICE, non_blocking=True) for m in batch['mask_labels']]
        class_labels = [c.to(DEVICE, non_blocking=True) for c in batch['class_labels']]

        optimizer.zero_grad(set_to_none=True)
        # No AMP here on purpose -- see markdown cell. Full precision only.
        outputs = model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            mask_labels=mask_labels,
            class_labels=class_labels,
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += float(loss.item())

        if batch_idx % PROGRESS_EVERY == 0 or batch_idx == len(train_loader):
            elapsed = time.perf_counter() - epoch_start
            rate = elapsed / batch_idx
            eta = rate * (len(train_loader) - batch_idx)
            print(f"  epoch {epoch:02d} batch {batch_idx}/{len(train_loader)} "
                  f"| avg loss={running_loss / batch_idx:.4f} "
                  f"| {rate:.2f}s/batch | elapsed={elapsed/60:.1f}m | ETA this epoch={eta/60:.1f}m",
                  flush=True)

    val_metrics = evaluate(val_loader)
    row = {'epoch': epoch, 'train_loss': running_loss / len(train_loader), **val_metrics}
    history.append(row)
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | loss={row['train_loss']:.4f} | val Dice={row['dice']:.4f} | val IoU={row['iou']:.4f} | val Recall={row['recall']:.4f}", flush=True)
    wandb.log(row, step=epoch)

    if row['dice'] > best_dice:
        best_dice = row['dice']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_metrics,
        }, RUN_DIR / 'best_model.pt')
    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(RUN_DIR / 'training_history.csv', index=False)
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation Dice was {best_dice:.4f} at epoch {best_epoch}.')
        break

wandb.summary['best_epoch'] = best_epoch
wandb.summary['best_validation_dice'] = best_dice
print('Best epoch:', best_epoch, 'Best validation Dice:', best_dice)

In [ ]:
# Load the best validation-Dice checkpoint and evaluate every fixed test subset.
checkpoint = torch.load(RUN_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

test_rows = []
for split_name, split_samples in test_sets.items():
    metrics = evaluate(make_loader(split_samples), measure_inference=True)
    test_rows.append({'split': split_name, **metrics})
    print(split_name, metrics)
    wandb.log({f'test_{split_name}_{key}': value for key, value in metrics.items()})

test_df = pd.DataFrame(test_rows)
display(test_df)

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(RUN_DIR / 'best_model.pt', FINAL_OUTPUT_DIR / 'best_model.pt')
shutil.copy2(RUN_DIR / 'training_history.csv', FINAL_OUTPUT_DIR / 'training_history.csv')
# Long-format, one row per split -- same shape as D-FINE's evaluation_metrics.csv.
test_df.to_csv(FINAL_OUTPUT_DIR / 'evaluation_metrics.csv', index=False)

by_split = {row['split']: row for row in test_rows}
overall = by_split['overall']

# Wide-format, one row per experiment -- matches the D-FINE/detection summary.csv schema
# exactly. mAP left blank: Mask2Former is instance-capable in general, but this run
# evaluates it in binary semantic mode for consistency with SegFormer/SegNeXt, and mAP
# isn't a valid metric for that mode (see project notes / markdown cell above for why).
summary_row = {
    'Experiment': RUN_NAME,
    'Model': MODEL_LABEL,
    'Batch': BATCH_SIZE,
    'Epochs': best_epoch,
    'mAP50': float('nan'),
    'mAP50_95': float('nan'),
    'Precision': overall['precision'],
    'Recall': overall['recall'],
    'mAP50_Small': float('nan'),
    'mAP50_Medium': float('nan'),
    'mAP50_Large': float('nan'),
    'Recall_Small': by_split['small']['recall'],
    'Recall_Medium': by_split['medium']['recall'],
    'Recall_Large': by_split['large']['recall'],
    'Inference_Time_ms': overall['inference_time_ms_per_image'],
    'FP_per_Image': overall['fp_per_image'],
    'Dice': overall['dice'],
    'IoU': overall['iou'],
    'Notes': f'{MODEL_LABEL} via HuggingFace transformers ({MODEL_NAME}), evaluated in binary semantic segmentation mode (not instance mode) for consistency with SegFormer/SegNeXt, full precision (no AMP -- DETR-style Hungarian matching, same risk class as D-FINE\'s documented AMP NaN issue), differential LR, fixed 640 split. mAP intentionally blank -- not computed in this semantic-mode run.',
}
summary_df = pd.DataFrame([summary_row])
display(summary_df)
summary_df.to_csv(FINAL_OUTPUT_DIR / 'summary.csv', index=False)

metadata = {
    'experiment': RUN_NAME,
    'model': MODEL_LABEL,
    'task': 'binary semantic defect segmentation',
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'early_stopping_patience': PATIENCE,
    'base_lr': BASE_LR,
    'backbone_lr': BACKBONE_LR,
    'amp': False,
    'best_epoch': best_epoch,
    'best_validation_dice': best_dice,
    'split_counts': {
        'train': len(train_samples), 'val': len(val_samples),
        'test': len(test_samples), 'test_small': len(test_sets['small']),
        'test_medium': len(test_sets['medium']), 'test_large': len(test_sets['large']),
    },
}
(FINAL_OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Saved model and all metrics to:', FINAL_OUTPUT_DIR)

wandb.finish()